# 🔥 La Guardiana del Fuego — Asistente experto en la saga Souls

**Proyecto Final — IA Generativa.** Agente RAG con **Google Gemini** (LLM + embeddings),
**ChromaDB** como base de conocimiento vectorial y **LangGraph** con memoria de conversación.

**Dominio:** lore, personajes, jefes, mecánicas e historia de desarrollo de Dark Souls I-III,
Demon's Souls, Bloodborne y Elden Ring (incluida *Shadow of the Erdtree*).

| Sección | Contenido |
|---|---|
| 0 | Configuración (funciona igual en Jupyter local y Google Colab) |
| 1 | Base documental |
| 2 | Base de conocimiento vectorial: chunking + ChromaDB + Gemini Embeddings |
| 3 | Verificación de la recuperación |
| 4 | System prompt |
| 5 | Agente en LangGraph |
| 6 | Demostración de memoria |
| 7 | Preguntas de ejemplo documentadas |
| 8 | Chat interactivo |

## 0. Configuración

La misma celda sirve para los dos entornos:
- **Colab:** clona el repositorio e instala dependencias. La API key se lee de *Secrets* (icono 🔑) con el nombre `GOOGLE_API_KEY`.
- **Jupyter local:** no hace nada extra. La API key se lee del archivo `.env` (ver `.env.example`).

In [ ]:
import os, sys

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/alvaromedinaro-png/souls-expert-rag.git"
    if not os.path.exists("/content/proyecto"):
        !git clone -q {REPO_URL} /content/proyecto
    %cd /content/proyecto
    !pip install -q -r requirements.txt

sys.path.insert(0, os.getcwd())

In [ ]:
from src.config import load_api_key, IN_COLAB, CHAT_MODEL, REWRITE_MODEL, EMBEDDING_MODEL

load_api_key()
print(f"Entorno: {'Google Colab' if IN_COLAB else 'Jupyter local'}")
print(f"LLM: {CHAT_MODEL} | Reescritura: {REWRITE_MODEL} | Embeddings: {EMBEDDING_MODEL}")

## 1. Base documental

Los documentos se descargaron con `src/download_data.py` desde las wikis de Fandom
(darksouls, bloodborne y eldenring), con licencia CC BY-SA, y desde Wikipedia (desarrollo, lanzamiento y recepción).
El script limpia el HTML: quita infoboxes, tablas, navegación, referencias y notas editoriales, y guarda cada página
como Markdown con una cabecera de metadatos (título, juego, URL y licencia).

Los datos ya vienen en `data/`. Para volver a descargarlos: `python -m src.download_data --force`.

In [ ]:
from collections import Counter
from src.ingest import load_documents

docs = load_documents()
total_chars = sum(len(d.page_content) for d in docs)
print(f"{len(docs)} documentos | {total_chars:,} caracteres (~{total_chars // 3000} páginas de texto)\n")
for game, n in Counter(d.metadata["game"] for d in docs).items():
    print(f"  {game:<12} {n:>3} documentos")

In [ ]:
print(docs[20].page_content[:800])

## 2. Base de conocimiento vectorial

**Chunking en dos niveles:**
1. `MarkdownHeaderTextSplitter` divide por secciones (Lore, Location, Background...) para que un chunk no mezcle temas.
2. `RecursiveCharacterTextSplitter` (1000 caracteres, solapamiento de 150) divide las secciones largas.

Cada chunk empieza con su **ruta de encabezados** (p. ej. `[Knight Artorias > Lore > Dark Souls]`).
Así el embedding sabe de quién habla el fragmento aunque el texto solo diga "he".

Los IDs de los chunks son deterministas (hash SHA-1), lo que hace la ingesta **reanudable** e idempotente:
si se corta por límite de cuota de la API, al volver a ejecutarla solo indexa lo que falta.

In [ ]:
from src.ingest import split_documents

chunks = split_documents(docs)
print(f"{len(chunks)} chunks\n")
print(chunks[123].page_content)
print(chunks[123].metadata)

In [ ]:
from src.ingest import ingest

ingest()  # crea/actualiza la colección en chroma_db/ con Gemini Embeddings

## 3. Verificación de la recuperación

Antes de conectar el agente comprobamos que la búsqueda semántica devuelve los fragmentos correctos.
La distancia es coseno: cuanto más baja, más parecido.

In [ ]:
from src.config import get_vectorstore

vectorstore = get_vectorstore()

test_queries = [
    "Who is Artorias and how was he corrupted by the Abyss?",
    "What is Insight in Bloodborne?",
    "Malenia Scarlet Rot curse",
    "¿Qué pasa si enlazas la Primera Llama?",  # en español: los embeddings son multilingües
]
for q in test_queries:
    print(f"\n🔎 {q}")
    for doc, score in vectorstore.similarity_search_with_score(q, k=3):
        print(f"   {score:.3f}  {doc.metadata['section'][:90]}")

## 4. System prompt

El prompt completo está en `src/prompts.py` y la justificación de cada decisión en el README. En resumen:

| Decisión | Motivo |
|---|---|
| Rol "La Guardiana del Fuego", tono cercano y evocador | Le da personalidad al asistente, sin sacrificar claridad (primero la respuesta) |
| Responder solo con el CONTEXTO | Evita alucinaciones: el lore Souls es críptico y fácil de inventar |
| Citas numeradas [n] | Hace verificable cada afirmación |
| Nombres propios en inglés | Las fuentes están en inglés y los nombres oficiales varían según la traducción |
| Admitir ambigüedad del lore | Gran parte del lore es interpretación de la comunidad |
| Mensaje fijo si no hay información y rechazo fuera de dominio | Comportamiento predecible ante preguntas sin respuesta |

Además hay un segundo prompt (`REWRITE_PROMPT`) para el nodo que reescribe la consulta.

In [ ]:
from src.prompts import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

## 5. Agente en LangGraph

```
START → rewrite_query → retrieve → generate → END
```

- **rewrite_query:** Gemini (`gemini-3.5-flash-lite`, un modelo ligero para una tarea sencilla), con salida estructurada (Pydantic), convierte la última pregunta y el historial en una
  **consulta autocontenida en inglés**: resuelve "él", "ese jefe"... y traduce a inglés, que es el idioma de los documentos.
  También detecta **el juego** para filtrar por metadatos.
- **retrieve:** busca los 6 chunks más parecidos en ChromaDB **en cada turno**, filtrando por juego
  (más los artículos generales de la saga) cuando se ha detectado uno.
- **generate:** Gemini responde con el system prompt, el contexto numerado y el historial reciente (últimos 12 mensajes).
- **Memoria:** el checkpointer `MemorySaver` guarda el estado de cada conversación por `thread_id`.

In [ ]:
from IPython.display import Image, display
from src.agent import build_agent, ask, chat, new_thread_id

agent = build_agent()
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:  # sin acceso a mermaid.ink
    print(agent.get_graph().draw_mermaid())

In [ ]:
from IPython.display import Markdown

def show(question, thread_id):
    r = ask(agent, question, thread_id)
    sources = "\n".join(f"- [{t}]({u})" for t, u in r["sources"])
    display(Markdown(
        f"**🧑 Pregunta:** {question}\n\n"
        f"*🔎 Búsqueda generada:* `{r['search_query']}` · *juego:* `{r['game'] or 'todos'}`\n\n"
        f"**🔥 Respuesta:**\n\n{r['answer']}\n\n**Fuentes:**\n{sources}\n\n---"
    ))

## 6. Demostración de memoria

En el mismo hilo (`thread_id`), las preguntas 2 y 3 **no nombran a Artorias**, solo dicen "su" y "él".
La búsqueda generada demuestra que el agente resuelve la referencia gracias a la memoria.

In [ ]:
memory_thread = new_thread_id()
show("¿Quién era Artorias?", memory_thread)
show("¿Y qué relación tenía con su lobo?", memory_thread)
show("¿Qué le pasó a él al final?", memory_thread)

**Contraprueba:** la misma pregunta en un hilo nuevo no tiene historial, así que el agente no sabe a quién se refiere "él".

In [ ]:
show("¿Qué le pasó a él al final?", new_thread_id())

In [ ]:
# El estado guardado por el checkpointer: todos los mensajes del hilo
for m in agent.get_state({"configurable": {"thread_id": memory_thread}}).values["messages"]:
    print(f"{m.type:>5}: {m.text[:100]}")

## 7. Preguntas de ejemplo documentadas

Cubren los tres juegos, una pregunta que mezcla juegos, una sin información en la base y una fuera de dominio.

### Dark Souls — lore

In [ ]:
show("¿Qué es la Edad del Fuego y por qué hay que enlazar la llama?", new_thread_id())

### Bloodborne — mecánica

In [ ]:
show("¿Qué es la Perspicacia (Insight) en Bloodborne y para qué sirve?", new_thread_id())

### Elden Ring — personaje

In [ ]:
show("¿Quién es Malenia y por qué sufre la Podredumbre Escarlata?", new_thread_id())

### Elden Ring — finales

In [ ]:
show("¿Qué finales tiene Elden Ring?", new_thread_id())

### Saga — desarrollo

In [ ]:
show("¿Quién es Hidetaka Miyazaki y qué es un juego soulslike?", new_thread_id())

### Sin información en la base

In [ ]:
show("¿Cuántos puntos de vida exactos tiene Malenia en NG+7?", new_thread_id())

### Fuera de dominio

In [ ]:
show("¿Me das una receta de paella?", new_thread_id())

## 8. Chat interactivo

Ejecuta la celda y escribe tus preguntas. Escribe `salir` para terminar. La conversación tiene memoria.

In [ ]:
chat(agent)

## Conclusiones

- **RAG con Gemini** funciona sobre 105 documentos (~780.000 caracteres) de tres wikis y Wikipedia.
- La **reescritura de la consulta** resuelve dos problemas a la vez: las preguntas en español frente a documentos en inglés,
  y las referencias a turnos anteriores ("él", "ese jefe").
- El **filtro por juego** evita mezclar conceptos homónimos (por ejemplo, los "finales" de cada juego).
- El **system prompt** limita las respuestas al contexto y obliga a citar, lo que reduce las alucinaciones.
- La interfaz web (`app.py`, Streamlit) reutiliza el mismo agente.